## Topic: Model Deployment (Flask API + ML Web App)
### Learning Objectives

By the end of today, you will:

- Understand model deployment
- Build a Flask API
- Connect ML model to a web interface
- Create a real-world project for portfolio

### PART 1 — Install Flask in Colab

Open a new cell and run:

In [ ]:
# Install required libraries for deployment
!pip install flask joblib numpy

! runs shell commands in Colab.

Installs Flask, joblib, numpy.

### 2. Create project files inside Colab
You can write files using Python’s %%writefile magic command.

Run new cell:

In [ ]:
# Create app.py
%%writefile app.py
from flask import Flask, request, jsonify, render_template
import joblib
import numpy as np

app = Flask(__name__)

# Load your trained model
model = joblib.load("model/churn_model.pkl")

@app.route("/")
def home():
    return render_template("index.html")

@app.route("/predict", methods=["POST"])
def predict():
    data = request.form.values()
    data = [float(x) for x in data]
    prediction = model.predict([data])
    return render_template("index.html", prediction_text=f"Prediction: {prediction[0]}")

if __name__ == "__main__":
    app.run(debug=True)


Writing app.py


In [ ]:
# Step 1: Create the templates folder
!mkdir -p templates

In [ ]:
%%writefile templates/index.html
<!DOCTYPE html>
<html>
<head>
    <title>ML App</title>
</head>
<body>
    <h2>Churn Prediction</h2>
    <form action="/predict" method="post">
        <input type="text" name="feature1" placeholder="Enter value" required>
        <input type="text" name="feature2" placeholder="Enter value" required>
        <button type="submit">Predict</button>
    </form>
    <h3>{{ prediction_text }}</h3>
</body>
</html>

Writing templates/index.html


👉 %%writefile saves the code into a file inside Colab.
👉 !mkdir -p templates creates the templates folder for HTML.

### 3. Run Flask app in Colab
Colab doesn’t allow direct localhost:5000 access. Instead, use ngrok to expose your Flask app.

### Option 1: Use pyngrok (recommended)

Run these steps in Colab:

1. Install ngrok binary

In [ ]:
!wget -q -O /usr/local/bin/ngrok https://bin.equinox.io/c/4VmDzA7iaHb/ngrok-stable-linux-amd64.zip
!unzip -o /usr/local/bin/ngrok -d /usr/local/bin/
!chmod +x /usr/local/bin/ngrok

Archive:  /usr/local/bin/ngrok
  inflating: /usr/local/bin/ngrok    


👉 This downloads and installs ngrok into Colab.

2. Add your authtoken

In [ ]:
# Run this once in a Colab cell (replace with your actual token):
!ngrok config add-authtoken 3CfgC9p6oHNdwzbu5nDqdmlxNRi_7nYRwZ3ceuEJJJVZeYerp

NAME:
   ngrok - tunnel local ports to public URLs and inspect traffic

DESCRIPTION:
    ngrok exposes local networked services behinds NATs and firewalls to the
    public internet over a secure tunnel. Share local websites, build/test
    webhook consumers and self-host personal services.
    Detailed help for each command is available with 'ngrok help <command>'.
    Open http://localhost:4040 for ngrok's web interface to inspect traffic.

EXAMPLES:
    ngrok http 80                    # secure public URL for port 80 web server
    ngrok http -subdomain=baz 8080   # port 8080 available at baz.ngrok.io
    ngrok http foo.dev:80            # tunnel to host:port instead of localhost
    ngrok http https://localhost     # expose a local https server
    ngrok tcp 22                     # tunnel arbitrary TCP traffic to port 22
    ngrok tls -hostname=foo.com 443  # TLS traffic for foo.com to port 443
    ngrok start foo bar baz          # start tunnels from the configuration file

VERSI

Replace YOUR_AUTHTOKEN_HERE with the token from your ngrok dashboard.

3. Run Flask with pyngrok

In [ ]:
!pip install pyngrok
!ngrok config add-authtoken 3CfgC9p6oHNdwzbu5nDqdmlxNRi_7nYRwZ3ceuEJJJVZeYerp

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
from flask import Flask
from pyngrok import ngrok

app = Flask(__name__)

@app.route("/")
def home():
    return "Hello, Flask is running!"

# Open a tunnel on port 5000
public_url = ngrok.connect(5000)
print("Public URL:", public_url)

app.run(port=5000)

Public URL: NgrokTunnel: "https://semisweet-snowstorm-mumps.ngrok-free.dev" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [21/Apr/2026 14:50:52] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [21/Apr/2026 14:50:53] "GET /favicon.ico HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [21/Apr/2026 14:51:09] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [21/Apr/2026 14:51:10] "GET /favicon.ico HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [21/Apr/2026 14:53:45] "GET /predict HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [21/Apr/2026 14:53:47] "GET /predict HTTP/1.1" 404 -


### Use the Ngrok public URL  
1. Instead of typing 127.0.0.1:5000 in your browser, copy the Ngrok link printed in the notebook output (e.g., https://semisweet-snowstorm-mumps.ngrok-free.dev).
That’s the correct address to access your Flask app from outside Colab.

2. Keep the Colab cell running  
Don’t stop or interrupt the notebook cell that runs app.run(port=5000). If the cell stops, the Ngrok tunnel closes and the link won’t work.

3. Check firewall/proxy settings (optional)  
If you’re running locally (not in Colab), then 127.0.0.1:5000 should work — but only if the Flask server is running on your machine. In that case, make sure nothing is blocking port 5000.

👉 This will give you a public URL (like https://xxxx.ngrok.io) that you can open in your browser.

👉 The form will appear, and you can test predictions.

### 4. Comments for Understanding

- app.py → Flask backend, loads model, defines routes.
- index.html → Frontend form, sends data to Flask.
- Colab specifics → Use %%writefile to create files, and flask-ngrok to expose the app.
- Model file → Must be saved in model/churn_model.pkl before running Flask.


###  Summary
- In Colab, you create files with %%writefile.
- Use flask-ngrok to run Flask apps and get a public link.
- Your workflow: Upload model → Create app.py → Create index.html →
 Run Flask with ngrok → Test predictions.
